In [19]:
import keras 
from keras.models import Sequential
from keras import layers, ops
from keras.optimizers import Adam
import numpy as np
from sklearn.metrics import classification_report
import joblib

In [21]:
# num_classes = 23 since the label has 23 unique labels
def build_classifier(input_dim = 122, num_classes = 23):
    model = Sequential(
        [
            keras.Input((input_dim, )),
            layers.Dense(64, activation = 'relu', kernel_initializer = 'he_normal',  kernel_regularizer = 'l2', name = 'hidden_layer1'),
            layers.Dropout(0.3),
            layers.Dense(32, activation = 'relu', kernel_initializer = 'he_normal',  kernel_regularizer = 'l2', name = 'hidden_layer2'),
            layers.Dropout(0.3),
            layers.Dense(16, activation = 'relu', kernel_initializer = 'he_normal',  kernel_regularizer = 'l2', name = 'hidden_layer3'),
            layers.Dropout(0.3),
            layers.Dense(num_classes, activation = 'softmax', name = 'softmax_layer')
        ],
        name = 'Classifier'
    )

    model.compile(
        optimizer = Adam(learning_rate = 0.01), 
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )


    return model

In [22]:
X_all = np.load("../data/X_train.npy")
Y_all = np.load("../data/Y_train.npy")
anomaly_flags = np.load("../data/anomaly_flags.npy")

X_flagged = X_all[anomaly_flags == 1]
Y_flagged = Y_all[anomaly_flags == 1]

model = build_classifier(input_dim = 122, num_classes = 23)

model.fit(X_flagged, Y_flagged, epochs = 20, batch_size = 128, validation_split = 0.1)

Epoch 1/20
369/369 ━━━━━━━━━━━━━━━━━━━━ 1s 623us/step - accuracy: 0.8924 - loss: 0.5920 - val_accuracy: 0.9423 - val_loss: 0.3347
Epoch 2/20
369/369 ━━━━━━━━━━━━━━━━━━━━ 0s 460us/step - accuracy: 0.9367 - loss: 0.3679 - val_accuracy: 0.9439 - val_loss: 0.2946
Epoch 3/20
369/369 ━━━━━━━━━━━━━━━━━━━━ 0s 454us/step - accuracy: 0.9439 - loss: 0.3418 - val_accuracy: 0.9691 - val_loss: 0.2742
Epoch 4/20
369/369 ━━━━━━━━━━━━━━━━━━━━ 0s 457us/step - accuracy: 0.9494 - loss: 0.3208 - val_accuracy: 0.9633 - val_loss: 0.2775
Epoch 5/20
369/369 ━━━━━━━━━━━━━━━━━━━━ 0s 458us/step - accuracy: 0.9499 - loss: 0.3165 - val_accuracy: 0.9645 - val_loss: 0.2673
Epoch 6/20
369/369 ━━━━━━━━━━━━━━━━━━━━ 0s 461us/step - accuracy: 0.9524 - loss: 0.3048 - val_accuracy: 0.9698 - val_loss: 0.2577
Epoch 7/20
369/369 ━━━━━━━━━━━━━━━━━━━━ 0s 462us/step - accuracy: 0.9537 - loss: 0.3014 - val_accuracy: 0.9744 - val_loss: 0.2414
Epoch 8/20
369/369 ━━━━━━━━━━━━━━━━━━━━ 0s 459us/step - accuracy: 0.9566 - loss: 0.2853 - 

In [24]:
# test the classifier

X_test = np.load("../data/X_test.npy")
Y_test = np.load("../data/Y_test.npy")


# predict
Y_pred = model.predict(X_test)
Y_pred_classes = np.argmax(Y_pred, axis=1)
Y_test_classes = np.argmax(Y_test, axis=1)

le = joblib.load("../data/label_encoder.pkl")

# get the unique classes present in test
labels_present = sorted(set(Y_test_classes) | set(Y_pred_classes))

# get only the corresponding names
names_present = [le.classes_[i] for i in labels_present]

print(classification_report(
    Y_test_classes,
    Y_pred_classes,
    labels=labels_present,
    target_names=names_present,
    zero_division=0
))


588/588 ━━━━━━━━━━━━━━━━━━━━ 0s 169us/step
                 precision    recall  f1-score   support

           back       0.00      0.00      0.00       359
buffer_overflow       0.00      0.00      0.00        20
      ftp_write       0.00      0.00      0.00         3
   guess_passwd       0.00      0.00      0.00      1231
           imap       0.00      0.00      0.00         1
        ipsweep       0.24      0.99      0.38       141
           land       0.00      0.00      0.00         7
     loadmodule       0.00      0.00      0.00         2
       multihop       0.00      0.00      0.00        18
        neptune       1.00      1.00      1.00      4657
           nmap       0.00      0.00      0.00        73
         normal       0.79      0.90      0.84      9711
           perl       0.00      0.00      0.00         2
            phf       0.00      0.00      0.00         2
            pod       0.00      0.00      0.00        41
      portsweep       0.56      0.96      0.

In [25]:
model.save('../models/classifier.keras')